In [ ]:
# === INJECTED CONFIG === (auto-filled by GitHub Actions — do not edit manually)
DB_HOST         = 'sql312.infinityfree.com'
DB_NAME         = 'if0_41143060_gpu'
DB_USER         = 'if0_41143060'
DB_PASS         = 'REPLACED_BY_GH_ACTIONS'
CF_TUNNEL_TOKEN = 'REPLACED_BY_GH_ACTIONS'
INTERNAL_TOKEN  = 'REPLACED_BY_GH_ACTIONS'
SITE_URL        = 'REPLACED_BY_GH_ACTIONS'


In [ ]:
# ── INSTALL DEPENDENCIES ──────────────────────────────────────
import subprocess, sys

def pip(pkg):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)

pip('pymysql')
pip('requests')
print('Dependencies installed')


In [ ]:
# ── IMPORTS & CONSTANTS ───────────────────────────────────────
import os, sys, time, json, signal, threading, subprocess
import requests, pymysql
from pathlib import Path
from datetime import datetime, timedelta

NUM_SLOTS    = 10
BASE_PORT    = 8800          # slot N uses port 8800+N
SESSION_MINS = 30
POLL_SECS    = 10            # how often orchestrator polls DB
WORK_DIR     = Path('/kaggle/working')
BOOT_TIME    = datetime.utcnow()

print(f'GPU Cloud Orchestrator — booting at {BOOT_TIME.isoformat()}')


In [ ]:
# ── DATABASE HELPERS ──────────────────────────────────────────
def get_db():
    return pymysql.connect(
        host=DB_HOST, user=DB_USER, password=DB_PASS,
        database=DB_NAME, charset='utf8mb4',
        cursorclass=pymysql.cursors.DictCursor,
        connect_timeout=10, autocommit=True
    )

def db_query(sql, params=()):
    for attempt in range(3):
        try:
            conn = get_db()
            with conn.cursor() as cur:
                cur.execute(sql, params)
                return cur.fetchall()
        except Exception as e:
            print(f'DB error (attempt {attempt+1}): {e}')
            time.sleep(2)
    return []

def db_exec(sql, params=()):
    for attempt in range(3):
        try:
            conn = get_db()
            with conn.cursor() as cur:
                cur.execute(sql, params)
            conn.commit()
            return True
        except Exception as e:
            print(f'DB exec error (attempt {attempt+1}): {e}')
            time.sleep(2)
    return False

print('DB helpers ready')


In [ ]:
# ── CLOUDFLARE TUNNEL SETUP ───────────────────────────────────
def install_cloudflared():
    cf_path = WORK_DIR / 'cloudflared'
    if cf_path.exists():
        print('cloudflared already present')
        return str(cf_path)

    print('Downloading cloudflared...')
    url = 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'
    subprocess.run(['wget', '-q', '-O', str(cf_path), url], check=True)
    subprocess.run(['chmod', '+x', str(cf_path)], check=True)
    print('cloudflared installed')
    return str(cf_path)

def write_cf_config():
    """Write cloudflared config with 10 ingress rules, one per slot."""
    # Read CF base domain from DB config
    rows = db_query("SELECT cfg_value FROM config WHERE cfg_key='cf_base_url'")
    cf_base = rows[0]['cfg_value'] if rows else 'https://slot{N}.yourdomain.com'

    ingress = []
    for i in range(1, NUM_SLOTS + 1):
        hostname = cf_base.replace('{N}', str(i)).replace('https://', '').replace('http://', '')
        ingress.append({
            'hostname': hostname,
            'service':  f'http://localhost:{BASE_PORT + i}'
        })
    ingress.append({'service': 'http_status:404'})  # catch-all required

    config = {
        'tunnel': CF_TUNNEL_TOKEN.split(':')[0] if ':' in CF_TUNNEL_TOKEN else 'gpu-cloud',
        'credentials-file': str(WORK_DIR / 'cf-creds.json'),
        'ingress': ingress
    }

    import yaml
    cfg_path = WORK_DIR / 'cf-config.yml'
    cfg_path.write_text(yaml.dump(config, default_flow_style=False))
    print(f'CF config written with {NUM_SLOTS} ingress rules')
    return str(cfg_path)

def start_cloudflared(cf_bin, cfg_path):
    proc = subprocess.Popen(
        [cf_bin, 'tunnel', '--config', cfg_path, 'run'],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    )
    time.sleep(5)
    if proc.poll() is not None:
        raise RuntimeError('cloudflared exited immediately — check tunnel token')
    print(f'cloudflared running (pid {proc.pid})')
    return proc

# Run tunnel setup
try:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml'], check=True)
    import yaml
    cf_bin  = install_cloudflared()

    # Write credentials file from token
    creds_path = WORK_DIR / 'cf-creds.json'
    creds_path.write_text(json.dumps({'AccountTag':'', 'TunnelSecret': CF_TUNNEL_TOKEN, 'TunnelID':''}))

    cf_cfg  = write_cf_config()
    cf_proc = start_cloudflared(cf_bin, cf_cfg)
    print('Cloudflare tunnel active')
except Exception as e:
    print(f'CF tunnel error: {e} — continuing without tunnel (dev mode)')
    cf_proc = None


In [ ]:
# ── SLOT MANAGER ──────────────────────────────────────────────
slot_procs = {}   # slot_id -> subprocess.Popen (Jupyter process)
slot_timers = {}  # slot_id -> threading.Timer

def setup_slot_dir(slot_id: int):
    slot_dir  = WORK_DIR / f'slot-{slot_id}'
    venv_dir  = slot_dir / 'venv'
    work_dir  = slot_dir / 'workspace'
    work_dir.mkdir(parents=True, exist_ok=True)

    if not (venv_dir / 'bin' / 'python').exists():
        print(f'[slot {slot_id}] creating venv...')
        subprocess.run([sys.executable, '-m', 'venv', str(venv_dir)], check=True)
        pip_bin = venv_dir / 'bin' / 'pip'
        subprocess.run([str(pip_bin), 'install', '-q',
                       'jupyter', 'notebook', 'ipykernel'], check=True)
        print(f'[slot {slot_id}] venv ready')
    return venv_dir, work_dir

def start_slot(slot_id: int, user_id: str):
    """Boot a Jupyter server for this slot and mark it active in DB."""
    print(f'[slot {slot_id}] starting for user {user_id}')
    venv_dir, work_dir = setup_slot_dir(slot_id)

    port  = BASE_PORT + slot_id
    token = f'slot{slot_id}tok{int(time.time())}'
    base_url = f'/slot/{slot_id}/'

    jupyter_bin = venv_dir / 'bin' / 'jupyter'
    cmd = [
        str(jupyter_bin), 'notebook',
        f'--port={port}',
        f'--NotebookApp.token={token}',
        f'--NotebookApp.base_url={base_url}',
        '--no-browser', '--ip=127.0.0.1',
        f'--notebook-dir={work_dir}',
        '--NotebookApp.allow_origin=*',
        '--NotebookApp.disable_check_xsrf=True',
    ]
    proc = subprocess.Popen(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    slot_procs[slot_id] = proc
    time.sleep(5)  # wait for Jupyter to bind

    if proc.poll() is not None:
        print(f'[slot {slot_id}] Jupyter failed to start!')
        db_exec('UPDATE slots SET status=%s WHERE slot_id=%s', ('free', slot_id))
        return

    # Get CF URL for this slot from config
    rows   = db_query("SELECT cfg_value FROM config WHERE cfg_key='cf_base_url'")
    cf_base = rows[0]['cfg_value'] if rows else 'http://localhost:{PORT}'
    cf_url  = cf_base.replace('{N}', str(slot_id))

    expires = (datetime.utcnow() + timedelta(minutes=SESSION_MINS)).strftime('%Y-%m-%d %H:%M:%S')

    # Update DB: slot is active
    db_exec(
        'UPDATE slots SET status=%s, user_id=%s, cf_url=%s, jupyter_token=%s, started_at=NOW(), expires_at=%s WHERE slot_id=%s',
        ('active', user_id, cf_url, token, expires, slot_id)
    )

    # Phone home to frontend so user sees their URL immediately
    try:
        requests.post(f'{SITE_URL}/queue.php', json={
            'action':         'slot_ready',
            'internal_token': INTERNAL_TOKEN,
            'slot_id':        slot_id,
            'cf_url':         cf_url,
            'jupyter_token':  token,
            'user_id':        user_id,
        }, timeout=10)
    except Exception as e:
        print(f'[slot {slot_id}] phone-home failed: {e}')

    # Arm 30-min kill timer
    t = threading.Timer(SESSION_MINS * 60, lambda: kill_slot(slot_id, 'timeout'))
    t.daemon = True
    t.start()
    slot_timers[slot_id] = t

    print(f'[slot {slot_id}] active — {cf_url} — expires {expires}')

def kill_slot(slot_id: int, reason: str = 'timeout'):
    """Kill a slot's Jupyter process, wipe workspace, free in DB."""
    print(f'[slot {slot_id}] killing — reason: {reason}')

    # Cancel timer if manually called
    if slot_id in slot_timers:
        slot_timers[slot_id].cancel()
        del slot_timers[slot_id]

    # Kill Jupyter
    if slot_id in slot_procs:
        try:
            slot_procs[slot_id].terminate()
            slot_procs[slot_id].wait(timeout=5)
        except Exception:
            pass
        del slot_procs[slot_id]

    # Wipe workspace
    workspace = WORK_DIR / f'slot-{slot_id}' / 'workspace'
    subprocess.run(['rm', '-rf', str(workspace)], check=False)
    workspace.mkdir(parents=True, exist_ok=True)

    # Log session to DB
    db_exec(
        '''INSERT INTO session_log (user_id, slot_id, started_at, ended_at, end_reason, duration_mins)
           SELECT user_id, slot_id, started_at, NOW(), %s,
                  TIMESTAMPDIFF(MINUTE, started_at, NOW())
           FROM slots WHERE slot_id=%s AND status='active'   ''',
        (reason, slot_id)
    )

    # Free slot in DB
    db_exec(
        'UPDATE slots SET status=%s, user_id=NULL, cf_url=NULL, jupyter_token=NULL, started_at=NULL, expires_at=NULL WHERE slot_id=%s',
        ('free', slot_id)
    )

    print(f'[slot {slot_id}] freed')

print('Slot manager ready')


In [ ]:
# ── GPU QUOTA TRACKER ─────────────────────────────────────────
quota_start_time = datetime.utcnow()

def get_used_minutes() -> int:
    elapsed = (datetime.utcnow() - quota_start_time).total_seconds() / 60
    # Add previously used minutes from DB
    rows = db_query('SELECT used_minutes FROM gpu_quota WHERE id=1')
    prev = int(rows[0]['used_minutes']) if rows else 0
    return prev + int(elapsed)

def quota_exhausted() -> bool:
    return get_used_minutes() >= 1800  # 30 hrs

def quota_warning() -> bool:
    return get_used_minutes() >= 1680  # 28 hrs — stop new queue joins

print('Quota tracker ready')


In [ ]:
# ── MAIN ORCHESTRATOR LOOP ────────────────────────────────────
print('Starting orchestrator...')

# Update DB: notebook is now running
db_exec('UPDATE gpu_quota SET notebook_status=%s, run_started_at=NOW(), last_heartbeat=NOW() WHERE id=1', ('running',))

# Pre-setup all venv dirs (can take ~5 min first time)
print('Pre-warming slot venvs (this takes a few minutes first time)...')
setup_threads = []
for sid in range(1, NUM_SLOTS + 1):
    t = threading.Thread(target=setup_slot_dir, args=(sid,), daemon=True)
    t.start()
    setup_threads.append(t)
for t in setup_threads:
    t.join()
print('All slot venvs ready')

HEARTBEAT_INTERVAL = 60   # seconds
last_heartbeat = time.time()

while True:
    try:
        used_mins = get_used_minutes()

        # ── HEARTBEAT every 60s ──────────────────────────────
        if time.time() - last_heartbeat >= HEARTBEAT_INTERVAL:
            nb_status = 'exhausted' if quota_exhausted() else 'running'
            db_exec(
                'UPDATE gpu_quota SET used_minutes=%s, notebook_status=%s, last_heartbeat=NOW() WHERE id=1',
                (used_mins, nb_status)
            )
            # Also phone home to frontend
            try:
                requests.post(f'{SITE_URL}/queue.php', json={
                    'action':          'heartbeat',
                    'internal_token':  INTERNAL_TOKEN,
                    'used_minutes':    used_mins,
                    'notebook_status': nb_status,
                }, timeout=8)
            except Exception:
                pass
            last_heartbeat = time.time()
            print(f'[heartbeat] {used_mins}/1800 min used — {nb_status}')

        # ── QUOTA EXHAUSTED: kill all active slots ────────────
        if quota_exhausted():
            print('[quota] 30 hrs exhausted — shutting down all slots')
            for sid in list(slot_procs.keys()):
                kill_slot(sid, 'quota')
            db_exec('UPDATE gpu_quota SET notebook_status=%s WHERE id=1', ('exhausted',))
            print('[quota] All slots freed. Notebook will idle until Kaggle 12hr kill.')
            time.sleep(300)  # sleep 5 min between checks
            continue

        # ── CHECK FOR NEW SLOT ASSIGNMENTS IN DB ──────────────
        booting = db_query(
            "SELECT slot_id, user_id FROM slots WHERE status='booting' AND user_id IS NOT NULL"
        )
        for row in booting:
            sid = int(row['slot_id'])
            uid = row['user_id']
            if sid not in slot_procs:  # not already starting
                print(f'[orchestrator] New assignment: slot {sid} → user {uid}')
                t = threading.Thread(target=start_slot, args=(sid, uid), daemon=True)
                t.start()

        # ── CHECK FOR MANUAL EARLY-END REQUESTS ──────────────
        # If a slot's user_id is NULL but status is still 'active' → user ended session via frontend
        orphaned = db_query(
            "SELECT slot_id FROM slots WHERE status='active' AND user_id IS NULL"
        )
        for row in orphaned:
            kill_slot(int(row['slot_id']), 'user_left')

        # ── CHECK FOR EXPIRED SESSIONS (belt-and-suspenders) ──
        expired = db_query(
            "SELECT slot_id FROM slots WHERE status='active' AND expires_at < NOW()"
        )
        for row in expired:
            sid = int(row['slot_id'])
            if sid not in slot_timers:  # timer didn't fire for some reason
                print(f'[orchestrator] Force-killing expired slot {sid}')
                kill_slot(sid, 'timeout')

        time.sleep(POLL_SECS)

    except KeyboardInterrupt:
        print('Orchestrator interrupted — cleaning up...')
        for sid in list(slot_procs.keys()):
            kill_slot(sid, 'killed')
        db_exec('UPDATE gpu_quota SET notebook_status=%s WHERE id=1', ('offline',))
        break
    except Exception as e:
        print(f'[orchestrator] Error: {e}')
        time.sleep(POLL_SECS)
